In [2]:
%run ./routing/10_logical_routing.ipynb

datasource='python_docs'
chain for python_docs


In [5]:
import numpy as np

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings


# Cosine similarity function
def cosine_similarity(a, b):
    """
    Cosine similarity calculate garne function
    """
    a = np.array(a)
    b = np.array(b)

    # Normalize vectors
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)

    # Cosine similarity matrix return garne
    return np.dot(a, b.T)


# Local embedding model load gareko
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")


# Physics ko prompt
physics_template = """
You are a very smart physics professor.

Answer physics questions in a simple and accurate way.

Question:
{query}
"""


# Mathematics ko prompt
math_template = """
You are a skilled mathematician.

Solve mathematical problems step by step.

Question:
{query}
"""


# Prompt haru ko embeddings banaeko
prompt_templates = [
    physics_template,
    math_template,
]

prompt_embeddings = embeddings.embed_documents(prompt_templates)


# Semantic router
def prompt_router(input):

    # User question ko embedding banaeko
    query_embedding = embeddings.embed_query(input["query"])

    # Prompt embeddings sanga similarity calculate gareko
    similarity = cosine_similarity(
        [query_embedding],
        prompt_embeddings,
    )[0]

    # Sabai bhanda similar prompt choose gareko
    most_similar = prompt_templates[np.argmax(similarity)]

    print("Using MATH" if most_similar == math_template else "Using PHYSICS")

    return PromptTemplate.from_template(most_similar)


# Local Ollama model
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)


# Semantic routing chain
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)


# Execute gareko
response = chain.invoke("What's a black hole?")

print(response)


Using PHYSICS
A great question to start with!

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape once it falls inside. It's formed when a massive star collapses under its own gravity and its density becomes so high that not even light can escape.

Imagine a cosmic sinkhole: anything that gets too close to the event horizon (the point of no return) will be pulled in by the intense gravitational force. The event horizon is like a boundary beyond which you're trapped forever, with no way out.

Here's why it's called "black": since not even light can escape, the black hole appears black because it doesn't emit any light or radiation. It's essentially invisible to us, except for its gravitational influence on nearby objects.

Now, I know what you're thinking: "But wait, if nothing escapes, how do we even detect black holes?" Ah, that's a great question! We can infer the presence of a black hole by observing the motion of s